# 02 - PyTorch CNN demo cho ClinVar sequence 201 bp

Notebook nay doc dataset da tao tu notebook 01:

- `processed/X_ref_alt_marker_201.npy`
- `processed/y.npy`
- `processed/clinvar_training_metadata.parquet`

Muc tieu: train nhanh mot CNN 1D bang PyTorch tren `ref_seq + alt_seq + mutation marker` 9-channel encoding de du doan `Y`:

- `1`: Pathogenic / Likely pathogenic
- `0`: Benign / Likely benign

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path("/mnt/MyData/Bioinformatics/Project")
PROCESSED_DIR = PROJECT_DIR / "processed"

X_PATH = PROCESSED_DIR / "X_ref_alt_marker_201.npy"
Y_PATH = PROCESSED_DIR / "y.npy"
METADATA_PATH = PROCESSED_DIR / "clinvar_training_metadata.parquet"

for path in [X_PATH, Y_PATH, METADATA_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Chua co file: {path}. Hay chay notebook 01 den buoc 12B sau khi them tensor ref/alt/marker."
        )

X_PATH, Y_PATH, METADATA_PATH

(PosixPath('/mnt/MyData/Bioinformatics/Project/processed/X_ref_alt_marker_201.npy'),
 PosixPath('/mnt/MyData/Bioinformatics/Project/processed/y.npy'),
 PosixPath('/mnt/MyData/Bioinformatics/Project/processed/clinvar_training_metadata.parquet'))

## 1. Load dataset da preprocess

`X` co shape `(n_variants, 201, 9)`.

9 channels gom:

- `0:4`: one-hot `ref_seq` theo thu tu A/C/G/T
- `4:8`: one-hot `alt_seq` theo thu tu A/C/G/T
- `8`: mutation marker, bang 1 tai vi tri center index 100

PyTorch `Conv1d` can input `(batch, channels, length)`, nen trong `Dataset` se transpose thanh `(9, 201)`.

In [2]:
X = np.load(X_PATH, mmap_mode="r")
y = np.load(Y_PATH)
metadata_df = pd.read_parquet(METADATA_PATH)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"metadata shape: {metadata_df.shape}")
print(f"X dtype: {X.dtype}")
print(f"y dtype: {y.dtype}")

assert X.ndim == 3
assert X.shape[1:] == (201, 9)
assert len(X) == len(y) == len(metadata_df)

pd.Series(y).value_counts().rename(index={0: "Benign/Likely benign", 1: "Pathogenic/Likely pathogenic"})

X shape: (460488, 201, 9)
y shape: (460488,)
metadata shape: (460488, 15)
X dtype: uint8
y dtype: int8


Benign/Likely benign            401054
Pathogenic/Likely pathogenic     59434
Name: count, dtype: int64

## 2. Tao train/validation/test split

Dung stratified split de giu ti le nhan `0/1` gan nhau giua cac tap.

In [3]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

indices = np.arange(len(y))
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y[temp_idx],
)

print(f"train: {len(train_idx):,}, labels={dict(zip(*np.unique(y[train_idx], return_counts=True)))}")
print(f"val:   {len(val_idx):,}, labels={dict(zip(*np.unique(y[val_idx], return_counts=True)))}")
print(f"test:  {len(test_idx):,}, labels={dict(zip(*np.unique(y[test_idx], return_counts=True)))}")

train: 322,341, labels={np.int8(0): np.int64(280737), np.int8(1): np.int64(41604)}
val:   69,073, labels={np.int8(0): np.int64(60158), np.int8(1): np.int64(8915)}
test:  69,074, labels={np.int8(0): np.int64(60159), np.int8(1): np.int64(8915)}


## 3. Tao PyTorch Dataset/DataLoader

In [4]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


class ClinVarSequenceDataset(Dataset):
    def __init__(self, X_array, y_array, indices):
        self.X_array = X_array
        self.y_array = y_array
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = self.indices[item]
        # X raw: (201, 9). Conv1d needs (channels, length) = (9, 201).
        # np.load(..., mmap_mode="r") returns read-only arrays, so copy before torch.from_numpy.
        x_np = np.asarray(self.X_array[idx]).T.copy()
        x = torch.from_numpy(x_np).float()
        target = torch.tensor(self.y_array[idx], dtype=torch.float32)
        return x, target


BATCH_SIZE = 512

train_loader = DataLoader(
    ClinVarSequenceDataset(X, y, train_idx),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    ClinVarSequenceDataset(X, y, val_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    ClinVarSequenceDataset(X, y, test_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

batch_x, batch_y = next(iter(train_loader))
print(batch_x.shape, batch_y.shape)

Device: cuda
torch.Size([512, 9, 201]) torch.Size([512])


## 4. Build CNN 1D PyTorch

Kien truc demo:

- Conv1d + BatchNorm + ReLU + MaxPool
- Conv1d + BatchNorm + ReLU + MaxPool
- Conv1d + BatchNorm + ReLU
- AdaptiveMaxPool
- Dense output logits

In [5]:
class ClinVarCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(9, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x).squeeze(1)


model = ClinVarCNN().to(DEVICE)
model


ClinVarCNN(
  (features): Sequential(
    (0): Conv1d(9, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): AdaptiveMaxPool1d(output_size=1)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.3, inplace=False)
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Dropout(p=0.2, inplace=Fa

## 5. Train nhanh

Dung `BCEWithLogitsLoss`. Neu class imbalance, `pos_weight` se tang/giam trong loss cho class `1`.

In [6]:
# PyTorch/sympy compatibility guard.
# In some long-running notebook kernels, sympy.core is not attached to the sympy module
# when torch.optim lazily imports torch._dynamo. Force-load it before creating optimizer.
import importlib
import sympy

if not hasattr(sympy, "core"):
    sympy.core = importlib.import_module("sympy.core")
if not hasattr(sympy.core, "symbol"):
    sympy.core.symbol = importlib.import_module("sympy.core.symbol")

positive_count = (y_train := y[train_idx]).sum()
negative_count = len(y_train) - positive_count
pos_weight = torch.tensor([negative_count / max(positive_count, 1)], dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(f"pos_weight: {pos_weight.item():.4f}")


pos_weight: 6.7478


In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score


def run_epoch(model, loader, train=False, epoch_label=""):
    model.train(train)
    total_loss = 0.0
    all_targets = []
    all_probs = []
    mode = "train" if train else "eval"

    progress = tqdm(loader, desc=f"{mode} {epoch_label}".strip(), unit="batch", leave=False)
    for batch_x, batch_y in progress:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)
        all_targets.append(batch_y.detach().cpu().numpy())
        progress.set_postfix(loss=f"{loss.item():.4f}")

    targets = np.concatenate(all_targets)
    probs = np.concatenate(all_probs)
    preds = (probs >= 0.5).astype(np.int8)

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": (preds == targets).mean(),
        "roc_auc": roc_auc_score(targets, probs),
        "pr_auc": average_precision_score(targets, probs),
    }
    return metrics, probs


EPOCHS = 5
best_val_auc = -np.inf
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch}/{EPOCHS}")
    train_metrics, _ = run_epoch(model, train_loader, train=True, epoch_label=f"{epoch}/{EPOCHS}")
    val_metrics, _ = run_epoch(model, val_loader, train=False, epoch_label=f"{epoch}/{EPOCHS}")

    row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

    if val_metrics["roc_auc"] > best_val_auc:
        best_val_auc = val_metrics["roc_auc"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"New best val_auc: {best_val_auc:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df

Epoch 1/5


train 1/5:   0%|          | 0/630 [00:00<?, ?batch/s]

eval 1/5:   0%|          | 0/135 [00:00<?, ?batch/s]

{'epoch': 1, 'train_loss': 1.1308, 'train_accuracy': np.float64(0.5668), 'train_roc_auc': 0.6547, 'train_pr_auc': 0.2435, 'val_loss': 0.9732, 'val_accuracy': np.float64(0.702), 'val_roc_auc': 0.7867, 'val_pr_auc': 0.4015}
New best val_auc: 0.7867
Epoch 2/5


train 2/5:   0%|          | 0/630 [00:00<?, ?batch/s]

...

{'epoch': 4, 'train_loss': 0.8567, 'train_accuracy': np.float64(0.753), 'train_roc_auc': 0.8428, 'train_pr_auc': 0.4898, 'val_loss': 0.8506, 'val_accuracy': np.float64(0.7674), 'val_roc_auc': 0.8451, 'val_pr_auc': 0.5125}
New best val_auc: 0.8451
Epoch 5/5


train 5/5:   0%|          | 0/630 [00:00<?, ?batch/s]

eval 5/5:   0%|          | 0/135 [00:00<?, ?batch/s]

{'epoch': 5, 'train_loss': 0.8254, 'train_accuracy': np.float64(0.7613), 'train_roc_auc': 0.8555, 'train_pr_auc': 0.5126, 'val_loss': 0.8531, 'val_accuracy': np.float64(0.7912), 'val_roc_auc': 0.8472, 'val_pr_auc': 0.5196}
New best val_auc: 0.8472


,epoch,train_loss,train_accuracy,train_roc_auc,train_pr_auc,val_loss,val_accuracy,val_roc_auc,val_pr_auc
0,1,1.130754,0.566825,0.654697,0.243475,0.973208,0.701982,0.786734,0.401521
1,2,0.953440,0.717408,0.797047,0.409298,0.916953,0.785068,0.819262,0.466043
2,3,0.895790,0.739171,0.825567,0.460290,0.879014,0.794609,0.838272,0.498887
3,4,0.856716,0.752976,0.842833,0.489847,0.850594,0.767377,0.845092,0.512476
4,5,0.825377,0.761333,0.855508,0.512632,0.853058,0.791221,0.847247,0.519593


## 6. Danh gia tren test set

In [8]:
from sklearn.metrics import classification_report, confusion_matrix

test_metrics, proba_test = run_epoch(model, test_loader, train=False, epoch_label="test")
pred_test = (proba_test >= 0.5).astype(np.int8)
y_test = y[test_idx]

print(test_metrics)
print("Classification report:")
print(classification_report(y_test, pred_test, target_names=["benign", "pathogenic"]))

confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, pred_test),
    index=["true_benign", "true_pathogenic"],
    columns=["pred_benign", "pred_pathogenic"],
)
confusion_matrix_df

eval test:   0%|          | 0/135 [00:00<?, ?batch/s]

{'loss': 0.8523833023192593, 'accuracy': np.float64(0.7933231027593596), 'roc_auc': 0.8481993804471991, 'pr_auc': 0.5223972538436702}
Classification report:
              precision    recall  f1-score   support

      benign       0.95      0.80      0.87     60159
  pathogenic       0.35      0.72      0.47      8915

    accuracy                           0.79     69074
   macro avg       0.65      0.76      0.67     69074
weighted avg       0.87      0.79      0.82     69074



,pred_benign,pred_pathogenic
true_benign,48350,11809
true_pathogenic,2467,6448


## 6B. Danh gia ket qua hien tai

Ket qua test cua model 9-channel `ref + alt + mutation marker`:

- `ROC-AUC ~= 0.848`: model phan biet pathogenic/benign kha tot cho baseline CNN dau tien.
- `PR-AUC ~= 0.522`: cao hon nhieu so voi baseline theo ti le pathogenic trong test set (`8915 / 69074 ~= 0.129`).
- `accuracy ~= 0.793`: tot hon ban `alt_seq` cu, nhung van khong nen xem la metric chinh vi dataset lech nhan.
- Pathogenic: `precision ~= 0.35`, `recall ~= 0.72`, `F1 ~= 0.47`.
- Benign: `precision ~= 0.95`, `recall ~= 0.80`, `F1 ~= 0.87`.

So voi model 4-channel chi dung `alt_seq` truoc do:

- ROC-AUC tang tu `~0.653` len `~0.848`.
- PR-AUC tang tu `~0.245` len `~0.522`.
- Pathogenic precision tang tu `~0.18` len `~0.35`.
- Pathogenic recall tang tu `~0.64` len `~0.72`.
- False positive benign -> pathogenic giam tu `26,449` xuong `11,809`.

Confusion matrix hien tai:

- True benign du doan dung: `48,350`
- True benign bi goi nham pathogenic: `11,809`
- True pathogenic bi bo sot: `2,467`
- True pathogenic du doan dung: `6,448`

Dien giai:

Y tuong `4 ref + 4 alt + 1 marker` co tac dung rat ro. Model khong con phai tu suy ra vi tri mutation va cung thay truc tiep su khac biet REF/ALT tai diem giua. Ket qua nay cho thay thong tin "dot bien thay doi base nao trong context nao" quan trong hon chi nhin chuoi `alt_seq` don le.

Model van con trade-off: recall pathogenic tot (`~72%`) nhung precision pathogenic moi `~35%`, nghia la van bao dong nham kha nhieu benign. Dieu nay chap nhan duoc cho baseline vi dang dung `pos_weight ~= 6.75`, loss uu tien khong bo sot pathogenic.

Buoc tiep theo nen lam:

1. Chon threshold theo PR curve thay vi co dinh `0.5` de dieu chinh precision/recall.
2. Train them epoch voi early stopping thuc su, vi val ROC-AUC van tang den epoch 5.
3. Thu split theo gene/chromosome de kiem tra leakage va do tong quat.
4. Them baseline logistic/MLP hoac CNN nho hon de xem CNN co that su can thiet khong.
5. Luu bang threshold/metrics de chon setting phu hop muc tieu: uu tien recall hay precision.

## 7. Xem mot vai prediction kem metadata

In [9]:
test_metadata_df = metadata_df.iloc[test_idx].copy()
test_metadata_df["pred_proba_pathogenic"] = proba_test
test_metadata_df["pred_label"] = pred_test

display(
    test_metadata_df[[
        "GeneSymbol",
        "ClinicalSignificance",
        "Y",
        "pred_proba_pathogenic",
        "pred_label",
        "Chromosome",
        "PositionVCF",
        "ReferenceAlleleVCF",
        "AlternateAlleleVCF",
        "ReviewStatus",
    ]]
    .sort_values("pred_proba_pathogenic", ascending=False)
    .head(20)
)

,GeneSymbol,ClinicalSignificance,Y,pred_proba_pathogenic,pred_label,Chromosome,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,ReviewStatus
301116,PCCA,Likely pathogenic,1,0.998540,1,13,100449306,G,T,"criteria provided, multiple submitters, no con..."
278951,TRAPPC9,Likely pathogenic,1,0.998423,1,8,140275657,C,A,"criteria provided, single submitter"
8665,BRCA1,Pathogenic/Likely pathogenic,1,0.998069,1,17,43115725,C,G,"criteria provided, multiple submitters, no con..."
178611,FLNA,Likely pathogenic,1,0.997851,1,X,154350030,C,G,"criteria provided, single submitter"
234200,EXT1,Pathogenic,1,0.997848,1,8,117835443,C,A,"criteria provided, single submitter"
291690,RPS6KA3,Likely pathogenic,1,0.997757,1,X,20195064,C,A,"criteria provided, single submitter"
58925,VPS13B,Pathogenic,1,0.997735,1,8,99720553,G,T,"criteria provided, multiple submitters, no con..."
32764,ASPM,Likely benign,0,0.997716,1,1,197105195,G,T,"criteria provided, single submitter"
34623,EFTUD2,Pathogenic,1,0.997290,1,17,44854268,C,A,"criteria provided, single submitter"
172387,COL11A1,Likely pathogenic,1,0.997075,1,1,102914351,C,G,"criteria provided, single submitter"


## 8. Luu model va ket qua demo

In [10]:
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

model_path = MODEL_DIR / "clinvar_ref_alt_marker_cnn_demo_pytorch.pt"
prediction_path = PROCESSED_DIR / "cnn_demo_ref_alt_marker_test_predictions_pytorch.parquet"
history_path = PROCESSED_DIR / "cnn_demo_ref_alt_marker_training_history_pytorch.parquet"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_class": "ClinVarCNN",
        "input_shape": (9, 201),
        "channels": ["ref_A", "ref_C", "ref_G", "ref_T", "alt_A", "alt_C", "alt_G", "alt_T", "mutation_marker"],
        "threshold": 0.5,
        "test_metrics": test_metrics,
    },
    model_path,
)

test_metadata_df.to_parquet(prediction_path, index=False, engine="pyarrow")
history_df.to_parquet(history_path, index=False, engine="pyarrow")

print(f"Saved model: {model_path}")
print(f"Saved predictions: {prediction_path}")
print(f"Saved history: {history_path}")

Saved model: /mnt/MyData/Bioinformatics/Project/models/clinvar_ref_alt_marker_cnn_demo_pytorch.pt
Saved predictions: /mnt/MyData/Bioinformatics/Project/processed/cnn_demo_ref_alt_marker_test_predictions_pytorch.parquet
Saved history: /mnt/MyData/Bioinformatics/Project/processed/cnn_demo_ref_alt_marker_training_history_pytorch.parquet
